# minGPT, as a short character model

This notebook walks a rewrite of [karpathy/minGPT](https://github.com/karpathy/minGPT). The code is `../src`. The smoke preset trains `gpt-micro` on Tiny Shakespeare. The cell below trains 100 steps of that same model.

## Embeddings

`GPT` adds two tables. `wte` is a vector per character. `wpe` is a learned vector per position, not a fixed sine. The output head `lm_head` is its own matrix. It does not share `wte`.

## Explicit attention

`CausalSelfAttention` maps each position to query, key, and value with one linear layer, then splits the heads. Scores are `q @ k` divided by sqrt(head size). A lower-triangular buffer fills future positions with `-inf` before the softmax. The heads are concatenated and passed through `c_proj`.

## NewGELU

The feed-forward network is 4x wider and uses the tanh approximation of GELU from GPT-2: `0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))`. Each block is pre-norm: LayerNorm, then the residual.

## Trainer

`Trainer` draws character chunks with replacement, clips gradients at 1.0, and steps AdamW. Weight decay 0.1 hits `Linear` weights. Biases, LayerNorm, and the embedding tables are left out. The device is CUDA, else MPS, else CPU. `pin_memory` is on only for CUDA.

## Character dataset

`CharDataset` builds the vocabulary from every character in `input.txt`. Each item is `block_size` characters and the next character at every position. The smoke uses block 64.

In [1]:
import sys
from pathlib import Path

EXPERIMENT = Path("..").resolve()
if str(EXPERIMENT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT))

from src.train import train

result = train(preset="smoke", max_iters=100, tag="mingpt-walkthrough")
print("device", result["device"])
print("parameters", result["num_parameters"])
print("last loss", result["history"][-1]["train_loss"])


model mingpt  preset smoke  device mps  type gpt-micro  parameters 809,856 transformer, 818,176 with head  steps 100  lr 0.0005


step 1: train loss 4.2144


step 50: train loss 2.6890


step 100: train loss 2.5307


sample:
ROMEO:
Hy watioor, ther teasth ad ad this theser,
Ans s thoutine hon hirone s to thoud hisesan ther te ors dor weresonistimy oungan,

Alieal has t,

Thoth ad minerend tame harrant by oun ad than othe he wys
wrote /Users/akashchauhan/Development/ai-models-learning/experiments/language/004-mingpt/runs/mingpt-walkthrough.pt  (2.8s)
device mps
parameters 818176
last loss 2.5306663513183594


## What the short run shows

The cell trained on MPS for 2.8 seconds. The model is `gpt-micro`: 809,856 parameters in the transformer and 818,176 once the untied head is counted. The logged training loss went from 4.214 at step 1 to 2.531 at step 100. That is one batch of characters, so the number jumps around, and the sample is still rough Shakespeare. The 400-step smoke in the README is the recorded run.